# 01 - Adaptive shots + keyframes + visual vectors (V2)

Legacy-compatible `shots.parquet`, `keyframes.parquet`, `visual.npy`, embedding records, shot vectors, WebP frames, and validation markers. OpenCLIP is encoded once.

**Execution model:** the notebook is a thin wrapper around the versioned script in `notebooks_V2/scripts/`. The cell is safe to rerun; completed videos are skipped by `_SUCCESS.json`. `RUN_ARGS` below defaults to a **full run** (no `--max-videos`/`--max-frames` cap). Swap in the smaller sample list first if you want to sanity-check on 1 video before spending full GPU time.


In [ ]:
%pip install -q av opencv-python-headless open_clip_torch pandas pyarrow pillow ffmpeg-python
!test -d /kaggle/working/TransNetV2 || GIT_LFS_SKIP_SMUDGE=1 git clone -q --depth 1 https://github.com/soCzech/TransNetV2.git /kaggle/working/TransNetV2
# The upstream repo's Git LFS bandwidth budget is exhausted (confirmed on a real Kaggle
# run), so inference/transnetv2-weights/* never comes down via plain `git clone`. Attach
# the "aic-ttvn-transnetv2-weights" Kaggle dataset (same files, mirrored) and copy them in.
!mkdir -p /kaggle/working/TransNetV2/inference/transnetv2-weights/variables
!test -f /kaggle/working/TransNetV2/inference/transnetv2-weights/saved_model.pb || cp /kaggle/input/aic-ttvn-transnetv2-weights/saved_model.pb /kaggle/working/TransNetV2/inference/transnetv2-weights/saved_model.pb
!test -f /kaggle/working/TransNetV2/inference/transnetv2-weights/variables/variables.index || cp /kaggle/input/aic-ttvn-transnetv2-weights/variables/variables.* /kaggle/working/TransNetV2/inference/transnetv2-weights/variables/


In [ ]:
from __future__ import annotations
import json
import runpy
import sys
from pathlib import Path

SCRIPT_NAME = '01_shot_keyframe_visual.py'
RUN_ARGS = []

def locate_v2_root() -> Path:
    candidates = [
        Path.cwd() / "pipelines" / "kaggle" / "notebooks_V2",
        Path.cwd() / "notebooks_V2",
        Path("/kaggle/working/notebooks_V2"),
    ]
    candidates.extend(path.parent.parent for path in Path("/kaggle/input").rglob("v2_runtime.py"))
    for candidate in candidates:
        if (candidate / "scripts" / SCRIPT_NAME).exists():
            return candidate
    raise FileNotFoundError("Attach the complete notebooks_V2 directory or run from the repository root")

V2_ROOT = locate_v2_root()
SCRIPT = V2_ROOT / "scripts" / SCRIPT_NAME
sys.path.insert(0, str(V2_ROOT / "lib"))
print("V2 root:", V2_ROOT)
print("Script:", SCRIPT)


In [ ]:
# Full run (RUN_ARGS above = []). To sample-test first, set RUN_ARGS = ["--max-videos", "1", "--no-make-compat-zips"] instead.
sys.argv = [str(SCRIPT), *RUN_ARGS]
runpy.run_path(str(SCRIPT), run_name="__main__")


In [ ]:
from pathlib import Path
import json

output_root = Path("/kaggle/working/aic-v2")
markers = list(output_root.rglob("_SUCCESS.json")) if output_root.exists() else []
print("Success markers:", len(markers))
for marker in markers[-10:]:
    try:
        payload = json.loads(marker.read_text(encoding="utf-8"))
        print(marker, payload.get("stage"), payload.get("video_id"), payload.get("success"))
    except Exception as exc:
        print("Invalid marker:", marker, exc)
